# 08 — Tomography of a Noisy Qubit

Close the density arc: reconstruct $\rho$ from real, noisy measurements and watch purity fall below $1$ — the honest reason the density matrix exists.

**Prerequisites:** `01/12_from_states_to_density_matrix`, `01/13_pure_vs_mixed`, `01/14_fidelity_and_trace_distance`.

**What you'll be able to do**
- Estimate a qubit's Bloch vector from $X$, $Y$, $Z$ measurements on a noisy backend.
- Reconstruct $\rho$ from those measurements (quantum state tomography).
- Explain why a real prepared state is mixed (purity $< 1$), not pure.

You built $\rho$ (`01/12`), separated pure from mixed (`01/13`), and learned to measure mixedness and distance (`01/14`). Now we point all of it at a real device: prepare $|+\rangle$, measure it, and rebuild its density matrix from the data — finding, honestly, that it is not pure. That is not a disappointment; it is the whole reason this object matters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.quantum import density

np.random.seed(42)
viz.use_course_style()

## 1. Tomography: a real (noisy) state is mixed

Let's *measure* a density matrix. We prepare $|+\rangle$, estimate the three Pauli expectations $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ on a **noisy** backend (carrying a real device's noise model), and reconstruct

$$ \rho = \tfrac12\big(I + \langle X\rangle X + \langle Y\rangle Y + \langle Z\rangle Z\big). $$

Each Pauli expectation is read off by rotating the chosen axis onto $Z$ before measuring — the standard single-qubit tomography recipe.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qot_course.hardware.runtime import get_noisy_backend

backend = get_noisy_backend()   # AerSimulator carrying a real device's noise model
SHOTS = 4096


def pauli_expectation_of_plus(basis):
    qc = QuantumCircuit(1, 1)
    qc.h(0)                                  # prepare |+>
    if basis == "X":
        qc.h(0)                              # rotate X-basis -> Z
    elif basis == "Y":
        qc.sdg(0); qc.h(0)                   # rotate Y-basis -> Z
    qc.measure(0, 0)
    counts = backend.run(transpile(qc, backend), shots=SHOTS, seed_simulator=42).result().get_counts()
    n0, n1 = counts.get("0", 0), counts.get("1", 0)
    return (n0 - n1) / (n0 + n1)


r = np.array([pauli_expectation_of_plus(b) for b in ("X", "Y", "Z")])
rho_hat = density.density_from_bloch(r)
print("estimated Bloch vector:", np.round(r, 3), " | |r| =", round(float(np.linalg.norm(r)), 3))
print("purity:", round(density.purity(rho_hat), 3), " (< 1  =>  the reconstructed state is MIXED)")
viz.plot_density_matrix(rho_hat, title="Tomography of a noisy |+>", basis_labels=["|0>", "|1>"])
plt.show()

**Read the figure and output.** The ideal $|+\rangle$ should have Bloch vector $(1, 0, 0)$ and purity $1$ — a point on the surface (notebook 07). The noisy device returns $|r| < 1$ and **purity below $1$**: the prepared state has drifted *inside* the Bloch ball. It is **mixed**. This is the honest, hands-on reason the density matrix exists — real states are never perfectly pure, and only $\rho$ can describe them.

## 2. Dictionary update

The density arc adds the entropy row: the quantum analogue of Shannon's entropy is von Neumann's, built from $\rho$'s eigenvalues.

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | Born rule $|\langle x|\psi\rangle|^2$ |
| Shannon entropy $H(p)$ | **von Neumann entropy** $S(\rho) = -\operatorname{tr}(\rho\log\rho)$ |

## Your turn

1. **More shots, less noise.** Re-run the tomography with `SHOTS = 16384`. Does $|r|$ move closer to $1$? Why does heavier averaging help here, exactly as it did for shot noise in notebook 04?
2. **How far did the noise push it?** Compute `density.trace_distance` between the ideal $|+\rangle$ and `rho_hat` (notebook 07's yardstick). How distinguishable did the noise make them?
3. **Which part decayed?** Compare `viz.plot_density_matrix` of the ideal $|+\rangle$ (notebook 05) with `rho_hat`: did the diagonal or the off-diagonal coherence shrink more? (You are previewing decoherence — the quantum channel of notebook 11.)

## What you built

- You estimated a qubit's Bloch vector from $X$, $Y$, $Z$ measurements on a noisy backend.
- You reconstructed $\rho$ from data and read its purity straight off.
- You saw a real prepared state is **mixed** (purity $< 1$) — the honest reason $\rho$ is indispensable.

That closes the **density arc**. From "what is $\rho$" to "here is a real, mixed $\rho$ measured off a device" — you now own the central object of the course in both theory and practice. Genuinely well done.

## What's next

One qubit has taught us everything it can. In `01/16_tensor_and_partial_trace` we put two systems together, and meet the quantum version of a marginal — the partial trace — which opens the door to entanglement.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 2 & 8, Cambridge University Press (2010).
- M. M. Wilde, *Quantum Information Theory*, ch. 4–5, Cambridge University Press (2017).

**Previous:** `notebooks/01_QuantumFoundations/14_fidelity_and_trace_distance.ipynb` · **Next:** `notebooks/01_QuantumFoundations/16_tensor_and_partial_trace.ipynb`